In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
sim_type = 'singleexon'
sensitivity = ''
padding = 0
cn_values = [3]

In [3]:
run_number = '29'
work_folder_extension = '_missed_2'
dnascreen_run = run_number + work_folder_extension
sim_dir='/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/simulation_hom_deletion/experiment_onsimulation_using_singlereg'
exome_depth_dir='/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/exome_depth'

In [4]:
selected_windows_dir = sim_dir + '/run' + dnascreen_run + '/selected_regions'

In [5]:
# Load the bed file into a dataframe
exons = pd.read_csv(exome_depth_dir + '/bed_files/9genes_25bp.fix.sorted.bed', sep='\t', header=None, names=['chr', 'start', 'end', 'name', 'idk','strand'])

## Checking exons covered by selected_windows (do we need this for single exon?)

In [6]:
# Check if the boundary of the simulated window falls within an exon for both VS and ED!

results = []

for cn in cn_values:
    selected_windows_file = selected_windows_dir + f'/selected_regions_cn{cn}_sample'
    selected_windows = pd.read_csv(selected_windows_file, sep='\t', 
                               header=None,
                               names=['sample', 'chr', 'start', 'end', 'gene'])
    
    # Doing this because we have repeated samples with different CN
    selected_windows['sample'] = 'cn' + str(cn) + '_' + selected_windows['sample'] + '.sorted'
    
    # Iterate over each row(sample) in the selected_windows dataframe
    for index, window in selected_windows.iterrows():
        # Get the chromosome, start, and end of the current window
        chr_window = window['chr']
        start_window = window['start']
        end_window = window['end']
        
        # Filter exons to find those that are on the same chromosome and within or overlap with the start and end of the current window
        exons_in_window = exons[(exons['chr'] == chr_window) &
                                (exons['start'] < end_window) &
                                (exons['end'] > start_window)]
        
        # Count the number of exons that match the criteria
        count_exons = len(exons_in_window)
        
        # Calculate the summed length of the exons
        summed_length_exons = (exons_in_window['end'] - exons_in_window['start']).sum()
        
        # Check if the start_window or end_window falls within any exons
        overlap_with_start = exons[(exons['chr'] == chr_window) &
                                   (exons['start'] <= start_window) &
                                   (exons['end'] >= start_window)]

        overlap_with_end = exons[(exons['chr'] == chr_window) &
                                 (exons['start'] <= end_window) &
                                 (exons['end'] >= end_window)]

        NA_dict = {'chr':'NA', 'start':'NA', 'end':'NA', 'name':'NA'}

        # Convert to list of dictionaries with required fields or 'NA' if no overlap
        overlap_with_start_dict = (overlap_with_start[['chr', 'start', 'end', 'name']]
                                   .drop_duplicates()
                                   .to_dict(orient='records')[0] if not overlap_with_start.empty else NA_dict)

        overlap_with_end_dict = (overlap_with_end[['chr', 'start', 'end', 'name']]
                                 .drop_duplicates()
                                 .to_dict(orient='records')[0] if not overlap_with_end.empty else NA_dict)
        
        # Append the result to the results list
        results.append({
            'cn': cn,
            'sample': window['sample'],
            'chr': chr_window,
            'start': start_window,
            'end': end_window,
            'gene': window['gene'],
            'count_exons': count_exons,
            'summed_length_exons': summed_length_exons,
            'overlap_with_start_chr': overlap_with_start_dict['chr'],
            'overlap_with_start_start': overlap_with_start_dict['start'],
            'overlap_with_start_end': overlap_with_start_dict['end'],
            'overlap_with_start_name': overlap_with_start_dict['name'],
            'overlap_with_end_chr': overlap_with_end_dict['chr'],
            'overlap_with_end_start': overlap_with_end_dict['start'],
            'overlap_with_end_end': overlap_with_end_dict['end'],
            'overlap_with_end_name': overlap_with_end_dict['name'],
        })


In [7]:
overlapping_exons = pd.DataFrame(results)

In [8]:
overlapping_exons.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/simulated_windows_info.csv')

## Storing per sample CNV calls for varseq

In [9]:
varseq_tables_path = sim_dir + '/run' + dnascreen_run + '/plot_generation/varseq_tables' + sensitivity
    
cnvcalls_path = os.path.join(varseq_tables_path, 'all_cnv_calls_sim_singleexon.tsv')
sample_path = os.path.join(varseq_tables_path, 'Varseq_sample_sim_singleexon.tsv')

cnvcalls_df = pd.read_csv(cnvcalls_path, sep='\t')
sample_df = pd.read_csv(sample_path, sep = '\t')

cov_sim_sample_dir = os.path.join(varseq_tables_path, 'cov_samples')

## The code below is a sanity check of whether samples in reference set are unique and not repeated. Also checks if the target sample is in the reference set itself. 

In [10]:
# Initialize counters and lists to store row indices
duplicate_rows = []
matching_rows = []

# Process each row in sample_df
for index, row in sample_df.iterrows():
    # Get main sample ID without prefix 'cn#_' and suffix '.sorted'
    main_sample_id = re.sub(r'^(cn\d+_)?|\.sorted$', '', row['Samples'])

    # Initialize lists to hold processed reference samples
    sample_names = []

    # Split the reference samples by commas
    for item in row['Reference Samples'].split(','):
        # Extract the sample name using regex
        name_match = re.search(r'^[^\(]+', item)
        
        if name_match:
            # Clean the reference sample name by removing '.hq.sorted.marked' and '.sorted'
            ref_sample_id = re.sub(r'\.hq\.sorted\.marked|\.sorted$', '', name_match.group().strip())
            sample_names.append(ref_sample_id)

    # Check for duplicate reference samples
    if len(sample_names) != len(set(sample_names)):
        duplicate_rows.append(index)  # Store the index of the row with duplicates

    # Check if any reference sample matches the main sample ID
    if main_sample_id in sample_names:
        matching_rows.append(index)  # Store the index of the row with matches

# Print the results
if duplicate_rows:
    print(f"Number of rows with duplicate reference samples: {len(duplicate_rows)}")
    print(f"Rows with duplicates: {duplicate_rows}")
else:
    print("No rows have duplicate reference samples.")

if matching_rows:
    print(f"Number of rows with reference samples matching main sample ID: {len(matching_rows)}")
    print(f"Rows with matches: {matching_rows}")
else:
    print("No rows have reference samples matching main sample ID.")

No rows have duplicate reference samples.
Number of rows with reference samples matching main sample ID: 2
Rows with matches: [0, 1]


## The code below extracts CNV calls per sample.

In [11]:
real_varseq_data_dir = "/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/VarSeq_CNV/varseq_tables_on_real_data" + "/run" + run_number
cov_real_sample_dir = real_varseq_data_dir + "/cov_samples"

In [12]:
import numpy as np
import pandas as pd
import os

# Storing average z-scores of non-matching exons for each copy number (CN)
avg_z_scores_all_nonmatching_sim_exons = [[] for _ in range(5)]

# Lists to store sample IDs, metrics for CNV calls, and metrics for coverage regions if the CNV is missed
sample_lst = []
percent_diff_lst = []
avg_nonmatching_sim_z_score_lst = []
CNV_coord_chr = []
CNV_coord_start = []
CNV_coord_end = []
region_size_lst = []
number_of_exons_varseq = []
type_lst = []
flag_lst = []
orig_target_mean_depth_lst = []
orig_z_score_lst = []
orig_ratio_lst = []
sim_target_mean_depth_lst = []
sim_z_score_lst = []
sim_ratio_lst = []
VS_target_mean_depth_lst = []
VS_z_score_lst = []
VS_ratio_lst = []
variants_considered_lst = []
supporting_LOH_variants_lst = []
estimated_CN_lst = []
GC_content_lst = []
p_val_lst = []
precision_levels_lst = []

# Iterate over each sample
for i in range(31, len(cnvcalls_df.columns), 12):
    sample_id = cnvcalls_df.columns[i][:-10]

    # Extract real sample id using re.search
    sample_id_real = re.search(r"(DNS-[A-Z0-9-]+_S[0-9]+)", sample_id).group(0)
    
    sample_cn = overlapping_exons[overlapping_exons['sample'] == sample_id]['cn'].values[0]
    percent_diff = sample_df[sample_df['Samples'] == sample_id]['Percent Difference'].values[0]
    
    # Load simulated coverage statistics
    cov_sim_sample_file = os.path.join(cov_sim_sample_dir, f'cov_sim_{sim_type} - {sample_id}.tsv')
    cov_sim_sample_df = pd.read_csv(cov_sim_sample_file, sep='\t')

    # Load original coverage statistics
    cov_real_sample_file = os.path.join(cov_real_sample_dir, f'CNV_run{run_number} - {sample_id_real}.tsv')
    cov_real_sample_df = pd.read_csv(cov_real_sample_file, sep='\t')

    # Check if there is a CNV call in VarSeq with p-value <= 0.05
    any_value_check = cnvcalls_df.iloc[0:len(cnvcalls_df), i+10].apply(
        lambda x: not pd.isnull(x) and (float(x) < 0.05)
    )

    # Define simulated CNV coordinates
    sim_chr = int(overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['chr'].values[0][3:])
    sim_start = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['start'].values[0]
    sim_end = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['end'].values[0]
    # count_exons_overlapping_simulated_window = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['count_exons_overlapping_simulated_window'].values[0]

    # Extract chromosome, start, and end from the Region column
    cov_sim_sample_df[['chr', 'start', 'end']] = cov_sim_sample_df['Region'].str.extract(r'(\d+):(\d+)-(\d+)')
    cov_sim_sample_df['chr'] = cov_sim_sample_df['chr'].astype(int)
    cov_sim_sample_df['start'] = cov_sim_sample_df['start'].astype(int)
    cov_sim_sample_df['end'] = cov_sim_sample_df['end'].astype(int)

    mask = (
        (cov_sim_sample_df['chr'] == sim_chr) &
        # checking for exon intervals that overlap or are within a simulated CNV
        ((((cov_sim_sample_df['start'] <= sim_end) &
         (cov_sim_sample_df['start'] >= sim_start)) |
        ((cov_sim_sample_df['end'] <= sim_end) &
         (cov_sim_sample_df['end'] >= sim_start))) | 
         # for simulated CNVs that are smaller than an exon interval
        (((cov_sim_sample_df['start'] <= sim_end) &
         (cov_sim_sample_df['start'] <= sim_start)) &
        ((cov_sim_sample_df['end'] >= sim_end) &
         (cov_sim_sample_df['end'] >= sim_start))))
    )

    cov_real_sample_df[['chr', 'start', 'end']] = cov_real_sample_df['Region'].str.extract(r'(\d+):(\d+)-(\d+)')
    cov_real_sample_df['chr'] = cov_real_sample_df['chr'].astype(int)
    cov_real_sample_df['start'] = cov_real_sample_df['start'].astype(int)
    cov_real_sample_df['end'] = cov_real_sample_df['end'].astype(int)

    mask = (
        (cov_real_sample_df['chr'] == sim_chr) &
        # checking for exon intervals that overlap or are within a simulated CNV
        ((((cov_real_sample_df['start'] <= sim_end) &
         (cov_real_sample_df['start'] >= sim_start)) |
        ((cov_real_sample_df['end'] <= sim_end) &
         (cov_real_sample_df['end'] >= sim_start))) | 
         # for simulated CNVs that are smaller than an exon interval
        (((cov_real_sample_df['start'] <= sim_end) &
         (cov_real_sample_df['start'] <= sim_start)) &
        ((cov_real_sample_df['end'] >= sim_end) &
         (cov_real_sample_df['end'] >= sim_start))))
    )


    # Extract non-matching exons' Z-Score and calculate average Z-score
    non_matching_sim_exons = cov_sim_sample_df[~mask].drop(columns=['chr', 'start', 'end'])
    z_score_non_matching_sim_exons = non_matching_sim_exons[f'Z Score for {sample_id}'].values
    avg_nonmatching_z_score_sim = np.mean(z_score_non_matching_sim_exons)
    avg_z_scores_all_nonmatching_sim_exons[sample_cn].extend(z_score_non_matching_sim_exons)


    # Extract metrics for matching exons in the simulated data
    matching_exons_sim = cov_sim_sample_df[mask].drop(columns=['chr', 'start', 'end'])
    mean_depth_sim = np.mean(matching_exons_sim[f'{sample_id} Mean Depth'])
    z_score_sim = np.mean(matching_exons_sim[f'Z Score for {sample_id}'])
    ratio_sim = np.mean(matching_exons_sim[f'Ratio for {sample_id}'])
    variants_considered = sum(matching_exons_sim[f'Variants Considered for {sample_id}'])

    # Extract metrics for matching exons in the original data
    matching_exons_orig = cov_real_sample_df[mask].drop(columns=['chr', 'start', 'end'])
    mean_depth_orig = np.mean(matching_exons_orig[f'{sample_id_real} Mean Depth'])
    z_score_orig = np.mean(matching_exons_orig[f'Z Score for {sample_id_real}'])
    ratio_orig = np.mean(matching_exons_orig[f'Ratio for {sample_id_real}'])
    # variants_considered = sum(matching_exons_sim[f'Variants Considered for {sample_id_real}'])

    if any(any_value_check):
        for v in range(0, len(any_value_check)):
            if any_value_check.iloc[v]:

                # Append sample-specific metrics
                sample_lst.append(sample_id)
                percent_diff_lst.append(percent_diff)
                avg_nonmatching_sim_z_score_lst.append(avg_nonmatching_z_score_sim)

                # Extract chr, start, and end of VarSeq calls
                chr = 'chr' + cnvcalls_df.iloc[v, 0].split(':')[0]
                start_end = cnvcalls_df.iloc[v, 0].split(':')[1]
                start = start_end.split('-')[0]
                end = start_end.split('-')[1]
                
                CNV_coord_chr.append(chr)
                CNV_coord_start.append(start)
                CNV_coord_end.append(end)
                region_size_lst.append(cnvcalls_df.iloc[v, 4])
                number_of_exons_varseq.append(cnvcalls_df.iloc[v, 2])
                type_lst.append(cnvcalls_df.iloc[v, i])
                flag_lst.append(cnvcalls_df.iloc[v, i+1])

                # Simulated region metrics
                sim_target_mean_depth_lst.append(mean_depth_sim)
                sim_z_score_lst.append(z_score_sim)
                sim_ratio_lst.append(ratio_sim)

                # Original region metrics
                orig_target_mean_depth_lst.append(mean_depth_orig)
                orig_z_score_lst.append(z_score_orig)
                orig_ratio_lst.append(ratio_orig)

                # VarSeq metrics
                VS_target_mean_depth_lst.append(cnvcalls_df.iloc[v, i+2])
                VS_z_score_lst.append(cnvcalls_df.iloc[v, i+3])
                VS_ratio_lst.append(cnvcalls_df.iloc[v, i+4])
                variants_considered_lst.append(variants_considered)
                supporting_LOH_variants_lst.append(cnvcalls_df.iloc[v, i+7])
                estimated_CN_lst.append(cnvcalls_df.iloc[v, i+5])
                GC_content_lst.append(cnvcalls_df.iloc[v, i+9])
                p_val_lst.append(cnvcalls_df.iloc[v, i+10])
                precision_levels_lst.append(cnvcalls_df.iloc[v, i+11])
    else:
        # Append metrics if no CNV call was made
        sample_lst.append(sample_id)
        percent_diff_lst.append(percent_diff)
        avg_nonmatching_sim_z_score_lst.append(avg_nonmatching_z_score_sim)

        CNV_coord_chr.append(np.nan)
        CNV_coord_start.append(np.nan)
        CNV_coord_end.append(np.nan)
        region_size_lst.append(np.nan)
        number_of_exons_varseq.append(np.nan)
        type_lst.append(np.nan)
        flag_lst.append(np.nan)

        # Simulated region metrics
        sim_target_mean_depth_lst.append(mean_depth_sim)
        sim_z_score_lst.append(z_score_sim)
        sim_ratio_lst.append(ratio_sim)

        # Original region metrics
        orig_target_mean_depth_lst.append(mean_depth_orig)
        orig_z_score_lst.append(z_score_orig)
        orig_ratio_lst.append(ratio_orig)

        # VarSeq metrics
        VS_target_mean_depth_lst.append(np.nan)
        VS_z_score_lst.append(np.nan)
        VS_ratio_lst.append(np.nan)
        variants_considered_lst.append(variants_considered)
        supporting_LOH_variants_lst.append(np.nan)
        estimated_CN_lst.append(np.nan)
        GC_content_lst.append(np.nan)
        p_val_lst.append(np.nan)
        precision_levels_lst.append(np.nan)


In [13]:
df_sample_CNV = pd.DataFrame({'sample':sample_lst, 
                              'Percent Difference':percent_diff_lst,
                              'Avg Z-score of Non-matching Exons':avg_nonmatching_sim_z_score_lst,
                              'VS_Call_chr':CNV_coord_chr,
                              'VS_Call_start':CNV_coord_start, 
                              'VS_Call_end':CNV_coord_end, 
                              'Size of CNV': region_size_lst,
                              'Number of Exons by VarSeq call':number_of_exons_varseq,
                                'Type of CNV':type_lst,
                              'VarSeq Flags for CNV': flag_lst,
                              'Target Mean Depth of Original CNV region': orig_target_mean_depth_lst,
                             'Z-Score of Original CNV region': orig_z_score_lst,
                             'Read Ratio of Original CNV region': orig_ratio_lst,
                              'Target Mean Depth of Simulated CNV': sim_target_mean_depth_lst,
                             'Z-Score of Simulated CNV': sim_z_score_lst,
                             'Read Ratio of Simulated CNV': sim_ratio_lst,
                             'VS Target Mean Depth of CNV': VS_target_mean_depth_lst,
                             'VS Z-Score of CNV': VS_z_score_lst,
                             'VS Read Ratio of CNV': VS_ratio_lst,
                              'Variants considered in CNV': variants_considered_lst,
                              'Supporting LOH variants in CNV': supporting_LOH_variants_lst,
                             'Estimated CN of CNV': estimated_CN_lst,
                             'GC Content of CNV': GC_content_lst,
                             'p-value of CNV':p_val_lst,
                             'Precision Level of call:':precision_levels_lst}
                            )

In [14]:
df_sample_CNV

,sample,Percent Difference,Avg Z-score of Non-matching Exons,VS_Call_chr,VS_Call_start,VS_Call_end,Size of CNV,Number of Exons by VarSeq call,Type of CNV,VarSeq Flags for CNV,...,Read Ratio of Simulated CNV,VS Target Mean Depth of CNV,VS Z-Score of CNV,VS Read Ratio of CNV,Variants considered in CNV,Supporting LOH variants in CNV,Estimated CN of CNV,GC Content of CNV,p-value of CNV,Precision Level of call:
0,cn3_DNS-XTHS-0105-C07-DNS010231_S51.sorted,4.50955,-0.034992,chr19,11116069,11116237,169,1,Duplicate,Insufficient Ratio,...,1.33832,433.609,7.49802,1.33832,1,0.0,3.0,0.544379,3.210405e-13,(1) Very High Sensitivity
1,cn3_DNS-XTHS-0106-E03-DNS009854_S117.sorted,4.34228,-0.085771,chr3,37028759,37028957,199,1,Duplicate,Insufficient Ratio,...,1.38393,891.779,14.88770,1.38393,0,0.0,3.0,0.472362,3.634752e-32,"(1) Very High Sensitivity,(2) High Sensitivity..."


In [15]:
# Find duplicate 'Sample' values
duplicate_samples = df_sample_CNV['sample'][df_sample_CNV['sample'].duplicated(keep=False)]

# Print or store the duplicate samples
print(duplicate_samples)

Series([], Name: sample, dtype: object)


In [16]:
VarSeq_results_df  = overlapping_exons.merge(df_sample_CNV, on='sample', how='left')

In [17]:
len(np.unique(VarSeq_results_df['sample']))

2

## Calculating the overlap percentage between the CNVs called and the simulated windows.

## The percentage is calculated based on the length of the 'Called CNV', not the 'Simulated CNV'.

In [18]:
# Outputting results of matching and percentage of CNV call that is within the simulated window
# we know some simulated windows have their boundaries fall within exons.
# So the CNV coordinates called won't be entirely within the simulated windows as the approach is a 
# Read-Depth based approach.

# Function to calculate overlap percentage
def calculate_within_window_percentage(start1, end1, start2, end2):
    overlap_start = max(start1, start2)
    overlap_end = min(end1, end2)
    overlap_length = max(0, overlap_end - overlap_start)
    cnvcall_length = end2 - start2
    return overlap_length / cnvcall_length if cnvcall_length > 0 else 0

# Iterate over the rows and calculate the overlap
def output_matching_and_within_window_percentage(row):
    # Extract chr, start, end from CNV_coordinates
    if not pd.isna(row['VS_Call_chr']):
        cnv_chr = int(row['VS_Call_chr'][3:])
        cnv_start = int(row['VS_Call_start'])
        cnv_end = int(row['VS_Call_end'])
        
        if cnv_chr != int(row['chr'][3:]):
            return 0
        else:
            overlap_percentage = calculate_within_window_percentage(row['start'], 
                                                                    row['end'], cnv_start, cnv_end)
            return overlap_percentage
    else:
        return np.nan

# Create the new column 'VarSeq_call_within_window'
VarSeq_results_df['VarSeq_overlap_within_window'] = VarSeq_results_df.apply(
    output_matching_and_within_window_percentage, axis=1)

In [19]:
VarSeq_results_df

,cn,sample,chr,start,end,gene,count_exons,summed_length_exons,overlap_with_start_chr,overlap_with_start_start,...,VS Target Mean Depth of CNV,VS Z-Score of CNV,VS Read Ratio of CNV,Variants considered in CNV,Supporting LOH variants in CNV,Estimated CN of CNV,GC Content of CNV,p-value of CNV,Precision Level of call:,VarSeq_overlap_within_window
0,3,cn3_DNS-XTHS-0105-C07-DNS010231_S51.sorted,chr19,11116068,11116237,LDLR_cds_10,1,169,chr19,11116068,...,433.609,7.49802,1.33832,1,0.0,3.0,0.544379,3.210405e-13,(1) Very High Sensitivity,1.0
1,3,cn3_DNS-XTHS-0106-E03-DNS009854_S117.sorted,chr3,37028758,37028957,MLH1_cds_12,1,199,chr3,37028758,...,891.779,14.88770,1.38393,0,0.0,3.0,0.472362,3.634752e-32,"(1) Very High Sensitivity,(2) High Sensitivity...",1.0


## Storing per sample RECALLED CNV calls for each sample-simulated CNV pair

In [20]:
def check_recalled(row):
    estimated_cn = row['Estimated CN of CNV']
    cn = row['cn']
    overlap = row['VarSeq_overlap_within_window']
    
    # Check the additional conditions and whether cn != 2
    # for singleexon CNVs if overlap is > 0, then it should be a 'hit'
    # SOMETHING WRONG WITH MY RECALL CHECK THE MISSED CNVS TABLE!!! okay found the culprit is cn=0 not all reads are removed.
    if cn != 2 and overlap > 0 and not pd.isna(overlap) and count_exons != 0:
        # Check if the CN values are the same, or the exception case for 3 and 4
        if estimated_cn == cn or (estimated_cn == 3 and cn == 4) or (estimated_cn == 4 and cn == 3):
            return True
    return False

# Apply the function to each row to create the 'recalled' column as boolean
VarSeq_results_df['recalled'] = VarSeq_results_df.apply(check_recalled, axis=1)

In [21]:
# Print out number of CNVs not covered by panel bed file
print("Number of CNVs not covered by Panel BED file: ", len(VarSeq_results_df[VarSeq_results_df['count_exons'] == 0]))

# Filter out rows where CNVs are not covered by the exon bed file:
VarSeq_results_df = VarSeq_results_df[VarSeq_results_df['count_exons'] != 0]

Number of CNVs not covered by Panel BED file:  0


In [22]:
# Step 1: Group by the combination of 'sample', 'chr', 'start', 'end' and check if any of them have 'recalled' == True
grouped = VarSeq_results_df.groupby(['cn', 'sample', 'chr', 'start', 'end'])['recalled'].any().reset_index()
number_of_exons_not_covered = len(overlapping_exons[overlapping_exons['count_exons'] == 0])

# Step 2: Filter out combinations where 'recalled' == True is present in the group
only_true_combinations = grouped[grouped['recalled'] == True][['cn', 'sample', 'chr', 'start', 'end']]

In [23]:
print('number of recalled CNVs: ' + \
      str(len(only_true_combinations)) + \
      ' out of ' + str((len(overlapping_exons[overlapping_exons['cn'] != 2]) - number_of_exons_not_covered)) + ' CNVs')

number of recalled CNVs: 2 out of 2 CNVs


## Parse the 'Precision Level of call:' column to separate the comma-separated values.

In [24]:
VarSeq_results_df['Precision Level of call:'] = VarSeq_results_df['Precision Level of call:'].str.strip()

# Define the precision levels with a regular expression
precision_levels = {
    'Very High Sensitivity': r"\(1\) Very High Sensitivity",
    'High Sensitivity': r"\(2\) High Sensitivity",
    'Balanced': r"\(3\) Balanced",
    'High Precision': r"\(4\) High Precision",
    'Very High Precision': r"\(5\) Very High Precision"
}

# Create new columns with True/False based on presence of each precision level
for level_name, pattern in precision_levels.items():
    VarSeq_results_df[level_name] = VarSeq_results_df['Precision Level of call:'].str.contains(pattern)

In [25]:
for level in precision_levels:
    print("Number of CNV calls made with " + level + ": " + str(len(VarSeq_results_df[VarSeq_results_df[level] == True])))

Number of CNV calls made with Very High Sensitivity: 2
Number of CNV calls made with High Sensitivity: 1
Number of CNV calls made with Balanced: 1
Number of CNV calls made with High Precision: 1
Number of CNV calls made with Very High Precision: 1


In [26]:
# Iterate through each precision level and check if it's the only True level in each row
for level in precision_levels:
    # Condition: 'level' is True and all other levels are False
    only_in_level = VarSeq_results_df[(VarSeq_results_df[level] == True) & 
                                      (VarSeq_results_df[[lvl for lvl in precision_levels if lvl != level]].sum(axis=1) == 0)]
    
    # Print the count of rows where the condition is met
    print("Number of CNV calls made only in " + level + ": " + str(len(only_in_level)))

Number of CNV calls made only in Very High Sensitivity: 1
Number of CNV calls made only in High Sensitivity: 0
Number of CNV calls made only in Balanced: 0
Number of CNV calls made only in High Precision: 0
Number of CNV calls made only in Very High Precision: 0


In [27]:
grouped = VarSeq_results_df.groupby(['cn', 'sample', 'chr', 'start', 'end']).any().reset_index()
recalled_cnvs = grouped[grouped['recalled'] == True]

for level in precision_levels:
    print("Number of Recalled CNVs detected with " + level + ": " + str(len(recalled_cnvs[(recalled_cnvs[level] == True)])))

Number of Recalled CNVs detected with Very High Sensitivity: 2
Number of Recalled CNVs detected with High Sensitivity: 1
Number of Recalled CNVs detected with Balanced: 1
Number of Recalled CNVs detected with High Precision: 1
Number of Recalled CNVs detected with Very High Precision: 1


In [28]:
VarSeq_results_df[VarSeq_results_df['recalled'] == True].to_csv(sim_dir + '/run' + \
                                                       dnascreen_run + \
                                                       '/plot_generation/Recalled_VS_CNV_Calls_metrics' + sensitivity + '.csv')

In [29]:
VarSeq_results_df.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/Full_VS_CNV_Calls_metrics' + sensitivity + '.csv')

## Listing down missed CNVs by VarSeq

In [30]:
# Step 1: Group by the combination of 'sample', 'chr', 'start', 'end' and check if any of them have 'recalled' == True
grouped = VarSeq_results_df.groupby(['cn', 'sample', 'chr', 'start', 'end'])['recalled'].any().reset_index()

# Step 2: Filter out combinations where 'recalled' == True is present in the group
only_false_combinations = grouped[(grouped['recalled'] == False) & (grouped['cn'] != 2)][['cn', 'sample', 'chr', 'start', 'end']]

# Step 3: Merge back with the original DataFrame to get only rows with 'recalled' == False
unique_false_rows = VarSeq_results_df.merge(only_false_combinations, on=['cn', 'sample', 'chr', 'start', 'end'])
unique_false_rows = unique_false_rows[unique_false_rows['recalled'] == False]

In [31]:
print('number of missed CNVs: ' + str(len(only_false_combinations)))

number of missed CNVs: 0


In [32]:
unique_false_rows.to_csv(varseq_tables_path + '/missed_cnvs_with_varseq_calls.csv')

In [33]:
# For easy exporting as a table to be presented
unique_false_rows[['cn', 'chr', 'start', 'end', 'gene', 'summed_length_exons', \
                   'Percent Difference', 'Read Ratio of Simulated CNV', 'Read Ratio of Original CNV region', 'Z-Score of Simulated CNV', 'Z-Score of Original CNV region', \
                   'Target Mean Depth of Original CNV region', 'Target Mean Depth of Simulated CNV']].to_csv(varseq_tables_path + '/notion_missed_cnvs_with_varseq_calls.csv')

# Exporting CNVs that are false positives to a separate csv table

In [34]:
# Create a new DataFrame for false positives
false_positives_varseq = VarSeq_results_df[VarSeq_results_df['VarSeq_overlap_within_window'] == 0]

# # Remove these rows from the original DataFrame
# VarSeq_results_df = VarSeq_results_df[VarSeq_results_df['VarSeq_overlap_within_window_based_on_window_length'] != 0]

# # Optionally, reset the index of the modified VarSeq_results_df
# VarSeq_results_df.reset_index(drop=True, inplace=True)

In [35]:
false_positives_varseq

,cn,sample,chr,start,end,gene,count_exons,summed_length_exons,overlap_with_start_chr,overlap_with_start_start,...,GC Content of CNV,p-value of CNV,Precision Level of call:,VarSeq_overlap_within_window,recalled,Very High Sensitivity,High Sensitivity,Balanced,High Precision,Very High Precision


In [36]:
false_positives_varseq.to_csv(varseq_tables_path + '/false_positives_varseq.csv')

## Extracting all Z-Scores of non-matching exons to plot mean and std error bars in Z-Score vs CN plot later

In [37]:
# Create a list to store the flattened Z-scores and their corresponding CN
z_scores_flattened = []
cn_values = []

for cn, z_scores in enumerate(avg_z_scores_all_nonmatching_sim_exons):
    z_scores_flattened.extend(z_scores)
    cn_values.extend([cn] * len(z_scores))

# Create a DataFrame from the flattened list
z_scores_per_cn_nonmatching_sim_exons = pd.DataFrame({
    'z_score_varseq': z_scores_flattened,
    'cn': cn_values
})


In [38]:
z_scores_per_cn_nonmatching_sim_exons.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/z_scores_per_cn_nonmatching_exons.csv')

## Checking if the reference set used in simulated set and the real data set is the same
## It won't be the same, this is because the simulated set is modified in one region and that will affect the correlation with neighbouring samples. So the reference set can be altered.

## Outputting all CN=2 simulations

In [39]:
cn_2_sims = VarSeq_results_df[VarSeq_results_df['cn'] == 2].drop_duplicates(subset='sample')

In [40]:
cn_2_sims.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/cn_2_sims.csv')

In [41]:
print("Number of CN=2 simulations falsely called: " + str(len(cn_2_sims[cn_2_sims['recalled'] == True])))

Number of CN=2 simulations falsely called: 0


## Storing per sample CNV calls for Exome Depth

In [42]:
# # Importing Exome Depth calls
# df_ED_CNV = pd.read_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/exome_depth_stats_different_ref_samples.csv')

In [43]:
# # Define the columns to concatenate
# columns_to_concat = [
#     'Frequency', 'ED_length', 'ED_chr', 'ED_start', 'ED_end', 'ED_percentage_within_window',
#     'Mean_Reads_Expected', 'Mean_Reads_Observed', 'Mean_Reads_Ratio', 'Mean_BF', 'Mean_Correlation'
# ]

# # Function to concatenate values for the specified columns
# def concatenate_values(group):
#     concatenated_values = {col: ', '.join(group[col].astype(str)) for col in columns_to_concat}
#     return pd.Series(concatenated_values)


In [44]:
# # Group by 'sample' and apply the concatenation function
# merged_df_ED_CNV = df_ED_CNV.groupby('sample').apply(concatenate_values).reset_index()

# # Drop duplicate rows and merge with concatenated values
# df_ED_CNV = df_ED_CNV.drop(columns=columns_to_concat).drop_duplicates(subset='sample')
# df_ED_CNV_final = pd.merge(df_ED_CNV, merged_df_ED_CNV, on='sample')

In [45]:
# df_ED_CNV_final

In [46]:
# merged_df = VarSeq_results_df.merge(df_ED_CNV_final, left_on=['sample', 'chr', 'start', 'end'], right_on=['sample', 'sim_chr', 'sim_start', 'sim_end'])

In [47]:
# # clean up the table by removing unecessary columns
# columns_to_drop = ['overlap_with_start_chr', 'overlap_with_start_start', 
#                    'overlap_with_start_end', 'overlap_with_start_name', 
#                    'overlap_with_end_chr', 'overlap_with_end_start', 
#                    'overlap_with_end_end', 'overlap_with_end_name',
#                    'call_key', 'sim_chr', 'sim_start', 
#                    'sim_end', 'sim_length', 'cn_ED'
#                   ]

# merged_df.drop(columns=columns_to_drop, inplace=True)

In [48]:
# merged_df.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/VS_ED_CNV_Calls_metrics.csv')

In [49]:
# merged_df